# SuperSimpleNet: efficiency and GPU energy of the selected checkpoint

This standalone notebook restores the already-selected **256 × 256, pose A, crop720, WideResNet-50-2** SuperSimpleNet run. Its synthetic training anomalies used Perlin threshold **0.2**, without a target-wheel restriction; image scoring uses the **top 1% mean of the native anomaly map**, with **σ4** smoothing of the output map.

It measures inference latency, CUDA allocator memory and **device-wide GPU energy per image**. It does not train, recalibrate, evaluate accuracy, repeat model selection or modify the checkpoint. The saved configuration and all state tensors are checked before inference; the current roadmap's restricted-anomaly preset is not substituted.

Use a fresh Kaggle **Tesla T4** session, the extracted synthetic dataset, and the same Google Drive secrets as the other notebooks. A trusted local final `model.ckpt` is also supported. The complete backbone is already saved in the checkpoint: no separate ImageNet weights download is needed.

The latency and energy helpers are the same explicit copies used for EfficientAD, TinyGLASS and PatchCore. Read the final JSON after the energy phase, not the earlier latency-only printout.


In [ ]:
from __future__ import annotations

import copy
import csv
import hashlib
import importlib.util
import inspect
import json
import math
import os
import platform
import random
import subprocess
import sys
import threading
import time
import warnings
from collections import Counter
from collections.abc import Iterable, Mapping
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Sequence

import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF
from torchvision.transforms.functional import gaussian_blur, pil_to_tensor
from tqdm.auto import tqdm


In [ ]:
from __future__ import annotations

MODEL_NAME = "supersimplenet"
CHECKPOINT_SOURCE = "google_drive"  # google_drive | local
MODEL_CHECKPOINT_PATH = None  # e.g. "/kaggle/input/my-supersimplenet-run/model.ckpt"
DRIVE_RUN_FOLDER_ID_OVERRIDE = None
FORCE_CHECKPOINT_DOWNLOAD = False
DRIVE_CHECKPOINT_CACHE_DIR = Path("/kaggle/working/supersimplenet_efficiency_checkpoints")
SELECTED_DRIVE_RUN = {
    "model": "supersimplenet",
    "folder_id": "1XPHgLw2jVMiRJRtwouqtIWFrQeEHDBeA",
    "checkpoint_file_id": "1uAdzWRKpmQZnjECnAuEXD08CtDY8xfhv",
    "comment": (
        "Selected local real-wheel catalog entry: Perlin 0.2, unrestricted "
        "synthetic anomalies, top-1% image score, sigma 4."
    ),
}

KAGGLE_INPUT = Path("/kaggle/input")
DATASET_ROOT_OVERRIDE = None  # optional extracted synthetic dataset directory
OUTPUT_ROOT = Path("/kaggle/working/supersimplenet_efficiency")
MODEL_INPUT_SIZE = (256, 256)
CAMERA_POSES = ("A_overhead",)
POSE_CROPS = {"A_overhead": (90, 240, 720, 720)}
EXPECTED_CHECKPOINT_SETTINGS = {
    "backbone": "wide_resnet50_2",
    "pretrained": True,
    "weights_name": "IMAGENET1K_V1",
    "layers": ["layer2", "layer3"],
    "input_size": [256, 256],
    "patch_size": 3,
    "perlin_threshold": 0.2,
    "restrict_synthetic_anomalies_to_target_mask": False,
    "gaussian_sigma": 4.0,
    "image_score_mode": "topk_fraction_mean",
    "image_score_fraction": 0.01,
    "stop_grad": True,
    "adapt_classification_features": False,
}
BENCHMARK_SPLIT = "validation"
BENCHMARK_BATCH_SIZE = 1
BENCHMARK_NUM_WORKERS = 0
BENCHMARK_PIN_MEMORY = False
BENCHMARK_WARMUP_RUNS = 10
BENCHMARK_TIMED_RUNS = 50
RANDOM_SEED = 42
REQUIRE_CUDA = True

if CHECKPOINT_SOURCE not in {"google_drive", "local"}:
    raise ValueError("CHECKPOINT_SOURCE must be google_drive or local")

# Separate sustained GPU energy measurement; same settings as the other models.
ENERGY_ENABLED = True
ENERGY_METHOD = "auto"  # NVML total-energy counter, or measured-power integration
ENERGY_REPEATS = 3
ENERGY_DURATION_SECONDS = 30.0
ENERGY_WARMUP_SECONDS = 5.0
ENERGY_POWER_SAMPLE_INTERVAL_SECONDS = 0.1


In [ ]:
from __future__ import annotations

# Installation is needed only for authenticated Drive downloads.
if CHECKPOINT_SOURCE == "google_drive":
    drive_requirements = {
        "google.oauth2.credentials": "google-auth",
        "googleapiclient": "google-api-python-client",
    }
    missing_packages = []
    for module_name, package_name in drive_requirements.items():
        try:
            installed = importlib.util.find_spec(module_name) is not None
        except ModuleNotFoundError:
            installed = False
        if not installed:
            missing_packages.append(package_name)
    if missing_packages:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

# Optional sensor dependency. Works in an older live session too.
# Installation/sensor failure never fabricates an energy estimate.
if globals().get("ENERGY_ENABLED", True) and importlib.util.find_spec("pynvml") is None:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nvidia-ml-py"])
    except Exception as error:
        print(f"WARNING: NVML bindings could not be installed; energy may be unavailable: {error}")


## Standalone definitions

The following cells explicitly duplicate the authoritative synthetic input pipeline, common model interface, SuperSimpleNet implementation and inference-efficiency helper under `src/anomaly_detection/`. Only repository-relative imports are replaced by earlier notebook definitions.

Training methods are retained to keep the model implementation synchronized, but no execution cell calls them. Initialization weights are disabled during construction; all learned and backbone tensors are restored strictly from the final checkpoint.


In [ ]:
from __future__ import annotations

RGBTriplet = tuple[float, float, float]
ImageSize = tuple[int, int]
SigmaRange = tuple[float, float]
CropBox = tuple[int, int, int, int]
IMAGENET_MEAN: RGBTriplet = (0.485, 0.456, 0.406)
IMAGENET_STD: RGBTriplet = (0.229, 0.224, 0.225)


@dataclass(frozen=True)
class PreprocessingConfig:
    """Optional preprocessing and train-time image augmentation settings.

    ``resize`` follows the torchvision convention ``(height, width)``. Every
    scalar augmentation value is disabled when ``None`` and otherwise denotes
    its maximum intensity. Photometric transforms affect RGB only.
    """

    resize: ImageSize | None = None
    resize_shorter_side: int | None = None
    center_crop: ImageSize | None = None
    pose_crop_enabled: bool = False
    pose_crops: Mapping[str, CropBox | None] = field(default_factory=dict)
    normalize_mean: RGBTriplet | None = None
    normalize_std: RGBTriplet | None = None
    augmentations_enabled: bool = True
    brightness: float | None = None
    contrast: float | None = None
    gamma: float | None = None
    saturation: float | None = None
    sensor_noise: float | None = None
    gaussian_noise: float | None = None
    gaussian_blur: SigmaRange | None = None

    def __post_init__(self) -> None:
        if self.resize is not None and self.resize_shorter_side is not None:
            raise ValueError("resize and resize_shorter_side are mutually exclusive")
        if self.resize is not None:
            if len(self.resize) != 2 or any(value < 1 for value in self.resize):
                raise ValueError("resize must be a positive (height, width) pair")
        if self.resize_shorter_side is not None and self.resize_shorter_side < 1:
            raise ValueError("resize_shorter_side must be positive")
        if self.center_crop is not None:
            if len(self.center_crop) != 2 or any(value < 1 for value in self.center_crop):
                raise ValueError("center_crop must be a positive (height, width) pair")
        for pose, crop in self.pose_crops.items():
            if not pose:
                raise ValueError("pose crop names cannot be empty")
            if crop is None:
                continue
            if len(crop) != 4:
                raise ValueError("pose crops must be (top, left, height, width)")
            if crop[2] < 1 or crop[3] < 1:
                raise ValueError("pose crop height and width must be positive")
            if crop[2] != crop[3]:
                raise ValueError("pose crops must be square")

        if (self.normalize_mean is None) != (self.normalize_std is None):
            raise ValueError("normalize_mean and normalize_std must be set together")
        if self.normalize_mean is not None:
            if len(self.normalize_mean) != 3 or len(self.normalize_std or ()) != 3:
                raise ValueError("normalization mean and std must contain three RGB values")
            if any(value <= 0 for value in self.normalize_std or ()):
                raise ValueError("normalization std values must be positive")

        for name in (
            "brightness",
            "contrast",
            "gamma",
            "saturation",
            "sensor_noise",
            "gaussian_noise",
        ):
            value = getattr(self, name)
            if value is not None and value < 0:
                raise ValueError(f"{name} must be non-negative or None")

        for name in ("brightness", "contrast", "gamma", "saturation"):
            value = getattr(self, name)
            if value is not None and value >= 1:
                raise ValueError(f"{name} must be smaller than 1")

        if self.gaussian_blur is not None:
            if len(self.gaussian_blur) != 2:
                raise ValueError("gaussian_blur must be a (min_sigma, max_sigma) pair")
            minimum, maximum = self.gaussian_blur
            if minimum <= 0 or maximum < minimum:
                raise ValueError("gaussian_blur requires 0 < min_sigma <= max_sigma")


class WheelPreprocessor:
    """Apply aligned geometric transforms and RGB-only photometric transforms."""

    def __init__(self, config: PreprocessingConfig | None = None) -> None:
        self.config = config or PreprocessingConfig()

    @staticmethod
    def _symmetric_factor(maximum_delta: float) -> float:
        return 1.0 + (2.0 * torch.rand(1).item() - 1.0) * maximum_delta

    @staticmethod
    def _odd_kernel_size(maximum_sigma: float) -> int:
        radius = max(1, round(3.0 * maximum_sigma))
        return 2 * radius + 1

    def __call__(
        self,
        image: torch.Tensor,
        target_mask: torch.Tensor,
        anomaly_mask: torch.Tensor,
        *,
        camera_pose: str | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        config = self.config

        if config.pose_crop_enabled:
            if camera_pose is None:
                raise ValueError(
                    "camera_pose is required when pose cropping is enabled"
                )
            if camera_pose not in config.pose_crops:
                raise ValueError(f"No pose crop configured for {camera_pose!r}")
            crop = config.pose_crops[camera_pose]
            if crop is not None:
                top, left, height, width = crop
                image_height, image_width = image.shape[-2:]
                if height > image_height or width > image_width:
                    raise ValueError(
                        f"Pose crop {crop!r} for {camera_pose!r} cannot fit "
                        f"inside the input image {(image_height, image_width)!r} "
                        "without padding"
                    )
                top = min(max(top, 0), image_height - height)
                left = min(max(left, 0), image_width - width)
                image = TF.crop(image, top, left, height, width)
                target_mask = TF.crop(target_mask, top, left, height, width)
                anomaly_mask = TF.crop(anomaly_mask, top, left, height, width)

        if config.resize is not None:
            image = TF.resize(
                image,
                config.resize,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize,
                interpolation=InterpolationMode.NEAREST,
            )
        elif config.resize_shorter_side is not None:
            image = TF.resize(
                image,
                config.resize_shorter_side,
                interpolation=InterpolationMode.BILINEAR,
                antialias=True,
            )
            target_mask = TF.resize(
                target_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )
            anomaly_mask = TF.resize(
                anomaly_mask,
                config.resize_shorter_side,
                interpolation=InterpolationMode.NEAREST,
            )

        if config.center_crop is not None:
            image = TF.center_crop(image, config.center_crop)
            target_mask = TF.center_crop(target_mask, config.center_crop)
            anomaly_mask = TF.center_crop(anomaly_mask, config.center_crop)

        # Every model-facing split uses one stable numerical contract.  Keeping
        # uint8 for evaluation while train-time augmentation returned float32
        # would either fail in the backbone or change the input scale by 255x.
        image = TF.convert_image_dtype(image, torch.float32)

        if config.augmentations_enabled:
            if config.brightness is not None:
                image = TF.adjust_brightness(image, self._symmetric_factor(config.brightness))
            if config.contrast is not None:
                image = TF.adjust_contrast(image, self._symmetric_factor(config.contrast))
            if config.gamma is not None:
                image = TF.adjust_gamma(image, self._symmetric_factor(config.gamma))
            if config.saturation is not None:
                image = TF.adjust_saturation(image, self._symmetric_factor(config.saturation))

            if config.gaussian_blur is not None:
                minimum, maximum = config.gaussian_blur
                sigma = minimum + torch.rand(1).item() * (maximum - minimum)
                kernel_size = self._odd_kernel_size(maximum)
                image = TF.gaussian_blur(image, kernel_size, sigma)

            if config.sensor_noise is not None and config.sensor_noise > 0:
                signal_scale = image.clamp(0.0, 1.0).sqrt()
                image = image + torch.randn_like(image) * signal_scale * config.sensor_noise
            if config.gaussian_noise is not None and config.gaussian_noise > 0:
                image = image + torch.randn_like(image) * config.gaussian_noise

        image = image.clamp(0.0, 1.0)
        if config.normalize_mean is not None:
            image = TF.normalize(
                image,
                mean=config.normalize_mean,
                std=config.normalize_std,
            )

        return image, target_mask, anomaly_mask

    def image_for_display(self, image: torch.Tensor) -> torch.Tensor:
        """Return a float RGB image in [0, 1], undoing normalization if needed."""
        if image.dtype == torch.uint8:
            return TF.convert_image_dtype(image, torch.float32)

        result = image.detach().clone()
        if self.config.normalize_mean is not None:
            mean = result.new_tensor(self.config.normalize_mean).view(3, 1, 1)
            std = result.new_tensor(self.config.normalize_std).view(3, 1, 1)
            result = result * std + mean
        return result.clamp(0.0, 1.0)


In [ ]:
from __future__ import annotations

REQUIRED_COLUMNS = {
    "image_id",
    "split",
    "condition",
    "image_path",
    "target_mask_path",
    "anomaly_mask_path",
    "pair_id",
    "camera_pose",
}
VALID_SPLITS = ("train", "validation", "test")
VALID_CONDITIONS = {"clean", "hole"}
ARTIFACT_PATH_FIELDS = ("image_path", "target_mask_path", "anomaly_mask_path")


class CuriosityWheelDataset(Dataset[dict[str, Any]]):
    """Load one split of the extracted Curiosity wheel Kaggle dataset."""

    def __init__(
        self,
        root: str | Path,
        split: str,
        preprocessing: WheelPreprocessor | None = None,
        camera_poses: Sequence[str] | None = None,
    ) -> None:
        self.root = Path(root).expanduser().resolve()
        self.split = split
        self.preprocessing = preprocessing or WheelPreprocessor()

        if split not in VALID_SPLITS:
            raise ValueError(f"Unsupported split {split!r}; expected one of {VALID_SPLITS}")
        if not self.root.is_dir():
            raise NotADirectoryError(f"Dataset root is not a directory: {self.root}")

        manifest_path = self.root / "samples.csv"
        if not manifest_path.is_file():
            raise FileNotFoundError(f"Dataset manifest is missing: {manifest_path}")

        with manifest_path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            columns = set(reader.fieldnames or [])
            missing = REQUIRED_COLUMNS - columns
            if missing:
                raise ValueError(f"samples.csv is missing required columns: {sorted(missing)}")
            rows = list(reader)

        self._validate_manifest_rows(rows)
        if isinstance(camera_poses, str):
            raise TypeError("camera_poses must be a sequence of pose names, not a string")
        if camera_poses is None:
            self.camera_poses = None
        else:
            self.camera_poses = tuple(dict.fromkeys(str(pose) for pose in camera_poses))
            if not self.camera_poses or any(not pose for pose in self.camera_poses):
                raise ValueError("camera_poses must contain at least one non-empty pose")
            available_poses = {row.get("camera_pose", "") for row in rows}
            unknown_poses = set(self.camera_poses) - available_poses
            if unknown_poses:
                raise ValueError(
                    f"Unknown camera poses: {sorted(unknown_poses)}; "
                    f"available poses: {sorted(available_poses)}"
                )

        self.rows = [
            row
            for row in rows
            if row["split"] == split
            and (
                self.camera_poses is None
                or row.get("camera_pose") in self.camera_poses
            )
        ]

        if not self.rows:
            raise ValueError(
                f"No samples found for split {split!r} and "
                f"camera_poses={self.camera_poses!r}"
            )

        for row in self.rows:
            for field in ARTIFACT_PATH_FIELDS:
                relative_path = row[field]
                if relative_path and not (self.root / relative_path).is_file():
                    raise FileNotFoundError(
                        f"Dataset artifact is missing: {self.root / relative_path}"
                    )

    @classmethod
    def _validate_manifest_rows(cls, rows: list[dict[str, str]]) -> None:
        """Validate the complete manifest before selecting one split."""
        if not rows:
            raise ValueError("samples.csv contains no samples")

        image_ids: set[str] = set()
        pairs: dict[str, list[dict[str, str]]] = {}
        for row in rows:
            image_id = row["image_id"]
            if not image_id:
                raise ValueError("samples.csv contains an empty image_id")
            if image_id in image_ids:
                raise ValueError(f"Duplicate image_id in samples.csv: {image_id}")
            image_ids.add(image_id)

            row_split = row["split"]
            if row_split not in VALID_SPLITS:
                raise ValueError(
                    f"Unsupported split {row_split!r} in sample {image_id}; "
                    f"expected one of {VALID_SPLITS}"
                )

            condition = row["condition"]
            if condition not in VALID_CONDITIONS:
                raise ValueError(f"Unsupported condition {condition!r} in sample {image_id}")
            if not row["image_path"] or not row["target_mask_path"]:
                raise ValueError(f"Sample has incomplete artifact paths: {image_id}")
            if condition == "hole" and not row["anomaly_mask_path"]:
                raise ValueError(f"Hole sample has no anomaly mask: {image_id}")
            if condition == "clean" and row["anomaly_mask_path"]:
                raise ValueError(f"Clean sample unexpectedly has an anomaly mask: {image_id}")
            for field in ARTIFACT_PATH_FIELDS:
                if row[field]:
                    cls._validate_relative_path(row[field])

            pair_id = row["pair_id"]
            if condition == "hole" and not pair_id:
                raise ValueError(f"Hole sample has no pair_id: {image_id}")
            if pair_id:
                pairs.setdefault(pair_id, []).append(row)

        for pair_id, pair_rows in pairs.items():
            conditions = {row["condition"] for row in pair_rows}
            splits = {row["split"] for row in pair_rows}
            if len(pair_rows) != 2 or conditions != VALID_CONDITIONS:
                raise ValueError(
                    f"Pair {pair_id!r} must contain exactly one clean and one hole sample"
                )
            if len(splits) != 1:
                raise ValueError(f"Pair {pair_id!r} crosses dataset splits: {sorted(splits)}")

    @staticmethod
    def _validate_relative_path(value: str) -> None:
        path = Path(value)
        if path.is_absolute() or ".." in path.parts:
            raise ValueError(f"Invalid dataset-relative path: {value!r}")

    def _load_image(self, relative_path: str, mode: str) -> Image.Image:
        path = self.root / relative_path
        if not path.is_file():
            raise FileNotFoundError(f"Dataset artifact is missing: {path}")
        with Image.open(path) as image:
            return image.convert(mode)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, index: int) -> dict[str, Any]:
        row = self.rows[index]
        image = pil_to_tensor(self._load_image(row["image_path"], "RGB"))
        target_mask = pil_to_tensor(self._load_image(row["target_mask_path"], "L"))

        if image.shape[1:] != target_mask.shape[1:]:
            raise ValueError(f"Image/target mask size mismatch for {row['image_id']}")

        anomaly_path = row["anomaly_mask_path"]
        if anomaly_path:
            anomaly_mask = pil_to_tensor(self._load_image(anomaly_path, "L"))
            if anomaly_mask.shape != target_mask.shape:
                raise ValueError(f"Target/anomaly mask size mismatch for {row['image_id']}")
        else:
            # Clean samples use an empty mask so every batch has one stable schema.
            anomaly_mask = torch.zeros_like(target_mask)

        image, target_mask, anomaly_mask = self.preprocessing(
            image,
            target_mask,
            anomaly_mask,
            camera_pose=row.get("camera_pose"),
        )

        return {
            "image": image,
            "target_mask": target_mask,
            "anomaly_mask": anomaly_mask,
            "label": torch.tensor(row["condition"] == "hole", dtype=torch.long),
            "has_anomaly_mask": bool(anomaly_path),
            "metadata": {
                key: value
                for key, value in row.items()
                if key not in {"image_path", "target_mask_path", "anomaly_mask_path"}
            },
        }


In [ ]:
from __future__ import annotations

# Kaggle exposes attached datasets as extracted directories.
def find_dataset_root(input_root: Path = KAGGLE_INPUT) -> Path:
    candidates = [
        manifest.parent
        for manifest in input_root.rglob("samples.csv")
        if (manifest.parent / "images").is_dir()
        and (manifest.parent / "masks").is_dir()
    ]
    if not candidates:
        raise FileNotFoundError(
            f"No valid dataset found under {input_root}: "
            "samples.csv, images/, and masks/ are required."
        )
    if len(candidates) > 1:
        raise RuntimeError(f"Multiple dataset candidates found: {candidates}")
    return candidates[0]

def build_dataloader(
    root: str | Path,
    split: str,
    *,
    batch_size: int = 4,
    num_workers: int = 0,
    pin_memory: bool = False,
    seed: int = 42,
    preprocessing: WheelPreprocessor | None = None,
    camera_poses: tuple[str, ...] | None = None,
) -> DataLoader:
    """Build one loader; only the training split is shuffled."""
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1")
    if num_workers < 0:
        raise ValueError("num_workers cannot be negative")

    dataset = CuriosityWheelDataset(
        root,
        split,
        preprocessing=preprocessing,
        camera_poses=camera_poses,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=split == "train",
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=num_workers > 0,
        generator=torch.Generator().manual_seed(seed),
    )


In [ ]:
from __future__ import annotations

CHECKPOINT_SCHEMA_VERSION = 2


@dataclass(frozen=True)
class AnomalyPrediction:
    """Model-independent anomaly output with normalized scores."""

    anomaly_score: torch.Tensor
    anomaly_map: torch.Tensor

    def __post_init__(self) -> None:
        if self.anomaly_score.ndim != 1:
            raise ValueError("anomaly_score must have shape [B]")
        if self.anomaly_map.ndim != 4 or self.anomaly_map.shape[1] != 1:
            raise ValueError("anomaly_map must have shape [B, 1, H, W]")
        if self.anomaly_score.shape[0] != self.anomaly_map.shape[0]:
            raise ValueError("anomaly_score and anomaly_map batch sizes must match")
        for name, value in (
            ("anomaly_score", self.anomaly_score),
            ("anomaly_map", self.anomaly_map),
        ):
            if not value.is_floating_point() or not torch.isfinite(value).all():
                raise ValueError(f"{name} must contain finite floating-point values")
            if value.numel() and (value.min() < 0 or value.max() > 1):
                raise ValueError(f"{name} must be normalized to [0, 1]")


class AnomalyDetector(nn.Module):
    """Small common interface shared by all anomaly detectors."""

    @property
    def is_fitted(self) -> bool:
        raise NotImplementedError

    def fit(
        self,
        train_loader: Iterable[Mapping[str, Any]],
        *,
        device: torch.device | str,
        validation_loader: Iterable[Mapping[str, Any]] | None = None,
        work_dir: str | Path | None = None,
        resume: bool = False,
    ) -> AnomalyDetector:
        raise NotImplementedError

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        raise NotImplementedError

    def checkpoint_config(self) -> dict[str, Any]:
        """Return behavior-affecting constructor settings for strict reloads."""
        return {}

    def save(self, path: str | Path, *, metadata: Mapping[str, Any] | None = None) -> Path:
        """Save model state and small run metadata in one checkpoint."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "schema_version": CHECKPOINT_SCHEMA_VERSION,
                "model_class": f"{type(self).__module__}.{type(self).__qualname__}",
                "model_config": self.checkpoint_config(),
                "model_state_dict": self.state_dict(),
                "metadata": dict(metadata or {}),
            },
            path,
        )
        return path

    def _prepare_state_dict_for_load(self, state_dict: Mapping[str, torch.Tensor]) -> None:
        """Allow models with dynamically sized buffers to prepare before load."""

    def load(
        self,
        path: str | Path,
        *,
        map_location: torch.device | str | None = None,
    ) -> dict[str, Any]:
        """Load a trusted project checkpoint and return its metadata."""
        payload = torch.load(Path(path), map_location=map_location)
        if not isinstance(payload, dict) or "model_state_dict" not in payload:
            raise ValueError("Checkpoint does not contain model_state_dict")
        if payload.get("schema_version") != CHECKPOINT_SCHEMA_VERSION:
            raise ValueError(
                f"Unsupported checkpoint schema {payload.get('schema_version')!r}; "
                f"expected {CHECKPOINT_SCHEMA_VERSION}"
            )
        expected_class = f"{type(self).__module__}.{type(self).__qualname__}"
        if payload.get("model_class") != expected_class:
            raise ValueError(
                f"Checkpoint model class {payload.get('model_class')!r} does not "
                f"match {expected_class!r}"
            )
        saved_config = payload.get("model_config")
        expected_config = self.checkpoint_config()
        if saved_config != expected_config:
            raise ValueError(
                "Checkpoint model configuration does not match the current model: "
                f"saved={saved_config!r}, current={expected_config!r}"
            )
        state_dict = payload["model_state_dict"]
        self._prepare_state_dict_for_load(state_dict)
        self.load_state_dict(state_dict)
        return dict(payload.get("metadata", {}))


In [ ]:
from __future__ import annotations

def _capture_training_rng_state(train_loader) -> dict[str, Any]:
    generator = getattr(train_loader, "generator", None)
    return {
        "torch_rng_state": torch.get_rng_state(),
        "python_rng_state": random.getstate(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
        "train_loader_generator_state": (
            generator.get_state() if generator is not None else None
        ),
    }


def _restore_training_rng_state(checkpoint: dict[str, Any], train_loader) -> None:
    generator_state = checkpoint.get("train_loader_generator_state")
    generator = getattr(train_loader, "generator", None)
    if generator_state is not None:
        if generator is None:
            raise TypeError("The train loader cannot restore its generator state")
        generator.set_state(
            torch.as_tensor(generator_state, dtype=torch.uint8, device="cpu")
        )
    torch.set_rng_state(
        torch.as_tensor(checkpoint["torch_rng_state"], dtype=torch.uint8, device="cpu")
    )
    if checkpoint.get("python_rng_state") is not None:
        random.setstate(checkpoint["python_rng_state"])
    cuda_rng_state = checkpoint.get("cuda_rng_state")
    if torch.cuda.is_available() and cuda_rng_state is not None:
        torch.cuda.set_rng_state_all([
            torch.as_tensor(state, dtype=torch.uint8, device="cpu")
            for state in cuda_rng_state
        ])


def _save_checkpoint_atomic(checkpoint: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)


def _clean_images(batch: Mapping[str, Any], device: torch.device) -> torch.Tensor:
    if "image" not in batch:
        raise KeyError("Every training batch must contain 'image'")
    if torch.any(torch.as_tensor(batch.get("label", 0)) != 0):
        raise ValueError("fit accepts only clean training images")
    return batch["image"].to(device, non_blocking=True)


def _image_auc(scores: list[float], labels: list[int]) -> float:
    positives = [score for score, label in zip(scores, labels) if label == 1]
    negatives = [score for score, label in zip(scores, labels) if label == 0]
    if not positives or not negatives:
        return float("nan")
    wins = sum(p > n for p in positives for n in negatives)
    ties = sum(p == n for p in positives for n in negatives)
    return (wins + 0.5 * ties) / (len(positives) * len(negatives))


@torch.no_grad()
def _validation_auc(
    model, loader, device: torch.device, max_batches: int | None,
) -> float:
    if loader is None:
        return float("nan")
    model.eval()
    scores: list[float] = []
    labels: list[int] = []
    for index, batch in enumerate(loader):
        if max_batches is not None and index >= max_batches:
            break
        prediction = model.predict(batch["image"].to(device, non_blocking=True))
        scores.extend(prediction.anomaly_score.cpu().tolist())
        labels.extend(torch.as_tensor(batch["label"]).int().tolist())
    return _image_auc(scores, labels)


In [ ]:
from __future__ import annotations

import copy
import math
from pathlib import Path
from typing import Any

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import get_model, get_model_weights
from torchvision.models.feature_extraction import create_feature_extractor
from torchvision.transforms.functional import gaussian_blur
from tqdm.auto import tqdm



# Ported from the MIT-licensed official implementation:
# https://github.com/blaz-r/SuperSimpleNet

def build_explicit_torchvision_backbone(
    backbone: str, *, pretrained: bool, weights_name: str,
) -> nn.Module:
    weights = None
    if pretrained:
        weights_enum = get_model_weights(backbone)
        try:
            weights = getattr(weights_enum, weights_name)
        except AttributeError as error:
            raise ValueError(
                f"Unknown torchvision weights {weights_name!r} for {backbone!r}"
            ) from error
    return get_model(backbone, weights=weights)


def _init_supersimplenet_weights(module: nn.Module) -> None:
    if isinstance(module, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_normal_(module.weight)
    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d)):
        nn.init.constant_(module.weight, 1)


def _rand_perlin_2d(
    shape: tuple[int, int], res: tuple[int, int], *, device: torch.device,
) -> torch.Tensor:
    """Torch port of the Perlin generator used by official SuperSimpleNet."""
    delta = (res[0] / shape[0], res[1] / shape[1])
    repeats = (shape[0] // res[0], shape[1] // res[1])
    y = torch.arange(0, res[0], delta[0], device=device)
    x = torch.arange(0, res[1], delta[1], device=device)
    grid = torch.stack(torch.meshgrid(y, x, indexing="ij"), dim=-1) % 1
    angles = 2 * math.pi * torch.rand(res[0] + 1, res[1] + 1, device=device)
    gradients = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)

    def tiled(y_slice, x_slice):
        return (
            gradients[y_slice[0]:y_slice[1], x_slice[0]:x_slice[1]]
            .repeat_interleave(repeats[0], 0)
            .repeat_interleave(repeats[1], 1)
        )

    def dot(gradient, shift):
        offsets = torch.stack(
            (grid[:shape[0], :shape[1], 0] + shift[0],
             grid[:shape[0], :shape[1], 1] + shift[1]),
            dim=-1,
        )
        return (offsets * gradient[:shape[0], :shape[1]]).sum(dim=-1)

    n00 = dot(tiled((0, -1), (0, -1)), (0, 0))
    n10 = dot(tiled((1, None), (0, -1)), (-1, 0))
    n01 = dot(tiled((0, -1), (1, None)), (0, -1))
    n11 = dot(tiled((1, None), (1, None)), (-1, -1))
    fade = lambda value: 6 * value**5 - 15 * value**4 + 10 * value**3
    blend = fade(grid[:shape[0], :shape[1]])
    return math.sqrt(2) * torch.lerp(
        torch.lerp(n00, n10, blend[..., 0]),
        torch.lerp(n01, n11, blend[..., 0]),
        blend[..., 1],
    )


def _ssn_focal_loss(
    probabilities: torch.Tensor, targets: torch.Tensor, gamma: float = 4.0,
    reduction: str | None = "mean",
) -> torch.Tensor:
    probabilities = probabilities.float()
    targets = targets.float()
    ce = F.binary_cross_entropy(probabilities, targets, reduction="none")
    pt = probabilities * targets + (1 - probabilities) * (1 - targets)
    loss = ce * (1 - pt).pow(gamma)
    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    return loss


class SuperSimpleFeatureExtractor(nn.Module):
    def __init__(
        self, *, backbone: str, pretrained: bool, weights_name: str,
        layers: tuple[str, ...], patch_size: int, input_size: tuple[int, int],
    ) -> None:
        super().__init__()
        model = build_explicit_torchvision_backbone(
            backbone, pretrained=pretrained, weights_name=weights_name
        )
        self.extractor = create_feature_extractor(
            model, return_nodes={layer: layer for layer in layers}
        )
        self.extractor.requires_grad_(False)
        self.layers = layers
        self.pooler = nn.AvgPool2d(patch_size, stride=1, padding=patch_size // 2)
        with torch.no_grad():
            outputs = self.extractor(torch.zeros(1, 3, *input_size))
        self.feature_channels = sum(value.shape[1] for value in outputs.values())

    def train(self, mode: bool = True):
        super().train(False)
        return self

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        self.extractor.eval()
        with torch.no_grad():
            features = list(self.extractor(images).values())
        height, width = features[0].shape[-2:]
        features = [
            F.interpolate(
                feature, size=(height * 2, width * 2),
                mode="bilinear", align_corners=True,
            )
            for feature in features
        ]
        return self.pooler(torch.cat(features, dim=1))


class SuperSimpleFeatureAdaptor(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.projection = nn.Conv2d(channels, channels, 1)
        self.apply(_init_supersimplenet_weights)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.projection(features)


class SuperSimpleDiscriminator(nn.Module):
    def __init__(self, channels: int, *, hidden_channels: int, stop_grad: bool) -> None:
        super().__init__()
        self.stop_grad = stop_grad
        self.segmentor = nn.Sequential(
            nn.Conv2d(channels, hidden_channels, 1),
            nn.BatchNorm2d(hidden_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(hidden_channels, 1, 1, bias=False),
        )
        self.decision_features = nn.Sequential(
            nn.Conv2d(channels + 1, 128, 5, padding="same"),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.score = nn.Linear(128 * 2 + 2, 1)
        self.apply(_init_supersimplenet_weights)

    def parameter_groups(self):
        return (
            self.segmentor.parameters(),
            [*self.decision_features.parameters(), *self.score.parameters()],
        )

    def forward(
        self, segmentation_features: torch.Tensor,
        classification_features: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        anomaly_map = self.segmentor(segmentation_features)
        decision_map = anomaly_map.detach() if self.stop_grad else anomaly_map
        decision = self.decision_features(
            torch.cat((classification_features, decision_map), dim=1)
        )
        map_for_pooling = decision_map if self.stop_grad else anomaly_map
        pooled = torch.cat((
            F.adaptive_max_pool2d(decision, 1),
            F.adaptive_avg_pool2d(decision, 1),
            F.adaptive_max_pool2d(map_for_pooling, 1),
            F.adaptive_avg_pool2d(map_for_pooling, 1),
        ), dim=1).flatten(1)
        return anomaly_map, self.score(pooled).flatten()


class SuperSimpleAnomalyGenerator(nn.Module):
    def __init__(
        self, *, noise_std: float, threshold: float,
        perlin_range: tuple[int, int] = (0, 6),
    ) -> None:
        super().__init__()
        self.noise_std = noise_std
        self.threshold = threshold
        self.perlin_range = perlin_range

    def _masks(
        self, batch_size: int, height: int, width: int, device: torch.device,
    ) -> torch.Tensor:
        power_height = 1 << (height - 1).bit_length()
        power_width = 1 << (width - 1).bit_length()
        masks = []
        for _ in range(batch_size):
            max_y = min(self.perlin_range[1], int(math.log2(power_height)) + 1)
            max_x = min(self.perlin_range[1], int(math.log2(power_width)) + 1)
            scale_y = 2 ** int(torch.randint(self.perlin_range[0], max_y, (1,)).item())
            scale_x = 2 ** int(torch.randint(self.perlin_range[0], max_x, (1,)).item())
            noise = _rand_perlin_2d(
                (power_height, power_width), (scale_y, scale_x), device=device
            )
            noise = F.interpolate(
                noise[None, None], size=(height, width),
                mode="bilinear", align_corners=False,
            )[0]
            mask = (noise > self.threshold).float()
            if torch.rand(()) > 0.5:
                mask.zero_()  # official no_anomaly="empty"
            masks.append(mask)
        return torch.stack(masks)

    def forward(
        self, features: torch.Tensor, adapted: torch.Tensor,
        target_masks: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch, _, height, width = adapted.shape
        features = torch.cat((features, features), dim=0)
        adapted = torch.cat((adapted, adapted), dim=0)
        mask = self._masks(batch * 2, height, width, adapted.device)
        if target_masks is not None:
            expected_shape = (batch, 1, height, width)
            if target_masks.shape != expected_shape:
                raise ValueError(
                    "SuperSimpleNet target masks must match the feature grid: "
                    f"expected {expected_shape}, got {tuple(target_masks.shape)}"
                )
            target_masks = (target_masks > 0).to(mask.dtype)
            if not target_masks.flatten(1).any(1).all():
                raise ValueError(
                    "SuperSimpleNet target masks must contain the target wheel"
                )
            mask = mask * torch.cat((target_masks, target_masks), dim=0)
        noise = torch.normal(0, self.noise_std, size=adapted.shape, device=adapted.device)
        return features + noise * mask, adapted + noise * mask, mask


class SuperSimpleNet(AnomalyDetector):
    """JIMS SuperSimpleNet in the official unsupervised MVTec configuration."""

    def __init__(
        self, *, backbone: str = "wide_resnet50_2",
        pretrained: bool = True, weights_name: str = "IMAGENET1K_V1",
        layers: tuple[str, ...] = ("layer2", "layer3"),
        input_size: tuple[int, int] = (256, 256), patch_size: int = 3,
        epochs: int = 300, noise_std: float = 0.015,
        perlin_threshold: float = 0.2, adaptor_learning_rate: float = 1e-4,
        segmentation_learning_rate: float = 2e-4,
        decision_learning_rate: float = 2e-4, scheduler_gamma: float = 0.4,
        stop_grad: bool = True, adapt_classification_features: bool = False,
        gradient_clip: bool = False, margin: float = 0.5,
        gaussian_sigma: float = 4.0, fixed_training_duration: bool = True,
        image_score_mode: str = "classification_head",
        image_score_fraction: float = 0.01,
        validation_interval: int = 4, validation_batches: int | None = 64,
        early_stopping_patience: int | None = 5,
        early_stopping_min_delta: float = 0.001,
        restrict_synthetic_anomalies_to_target_mask: bool = False,
        max_samples_per_epoch: int | None = None, checkpoint_interval: int = 1,
    ) -> None:
        super().__init__()
        self.backbone_name = backbone
        self.pretrained = pretrained
        self.weights_name = weights_name
        self.layers = tuple(layers)
        self.input_size = tuple(input_size)
        self.patch_size = patch_size
        self.epochs = epochs
        self.noise_std = noise_std
        self.perlin_threshold = perlin_threshold
        self.adaptor_learning_rate = adaptor_learning_rate
        self.segmentation_learning_rate = segmentation_learning_rate
        self.decision_learning_rate = decision_learning_rate
        self.scheduler_gamma = scheduler_gamma
        self.stop_grad = stop_grad
        self.adapt_classification_features = adapt_classification_features
        self.gradient_clip = gradient_clip
        self.margin = margin
        self.gaussian_sigma = gaussian_sigma
        if image_score_mode not in {"classification_head", "topk_fraction_mean"}:
            raise ValueError(
                "image_score_mode must be classification_head or "
                "topk_fraction_mean"
            )
        if not 0 < image_score_fraction <= 1:
            raise ValueError("image_score_fraction must be in (0, 1]")
        self.image_score_mode = image_score_mode
        self.image_score_fraction = float(image_score_fraction)
        self.fixed_training_duration = fixed_training_duration
        if validation_interval < 1:
            raise ValueError("validation_interval must be at least 1")
        self.validation_interval = validation_interval
        self.validation_batches = validation_batches
        if early_stopping_patience is not None and early_stopping_patience < 1:
            raise ValueError("early_stopping_patience must be positive or None")
        if early_stopping_min_delta < 0:
            raise ValueError("early_stopping_min_delta cannot be negative")
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_min_delta = early_stopping_min_delta
        self.restrict_synthetic_anomalies_to_target_mask = bool(
            restrict_synthetic_anomalies_to_target_mask
        )
        self.max_samples_per_epoch = max_samples_per_epoch
        if checkpoint_interval < 1:
            raise ValueError("checkpoint_interval must be at least 1")
        self.checkpoint_interval = checkpoint_interval

        self.features = SuperSimpleFeatureExtractor(
            backbone=backbone, pretrained=pretrained, weights_name=weights_name,
            layers=self.layers, patch_size=patch_size, input_size=self.input_size,
        )
        channels = self.features.feature_channels
        self.adaptor = SuperSimpleFeatureAdaptor(channels)
        self.discriminator = SuperSimpleDiscriminator(
            channels, hidden_channels=1024, stop_grad=stop_grad
        )
        self.anomaly_generator = SuperSimpleAnomalyGenerator(
            noise_std=noise_std, threshold=perlin_threshold
        )
        self.register_buffer("fitted", torch.tensor(False))
        self.fit_summary: dict[str, Any] = {}

    @property
    def is_fitted(self) -> bool:
        return bool(self.fitted)

    def train(self, mode: bool = True):
        super().train(mode)
        self.features.eval()
        return self

    def checkpoint_config(self) -> dict[str, Any]:
        return {
            "backbone": self.backbone_name, "pretrained": self.pretrained,
            "weights_name": self.weights_name, "layers": list(self.layers),
            "input_size": list(self.input_size), "patch_size": self.patch_size,
            "epochs": self.epochs, "noise_std": self.noise_std,
            "perlin_threshold": self.perlin_threshold,
            "adaptor_learning_rate": self.adaptor_learning_rate,
            "segmentation_learning_rate": self.segmentation_learning_rate,
            "decision_learning_rate": self.decision_learning_rate,
            "scheduler_gamma": self.scheduler_gamma, "stop_grad": self.stop_grad,
            "adapt_classification_features": self.adapt_classification_features,
            "gradient_clip": self.gradient_clip, "margin": self.margin,
            "gaussian_sigma": self.gaussian_sigma,
            "image_score_mode": self.image_score_mode,
            "image_score_fraction": self.image_score_fraction,
            "fixed_training_duration": self.fixed_training_duration,
            "validation_interval": self.validation_interval,
            "validation_batches": self.validation_batches,
            "early_stopping_patience": self.early_stopping_patience,
            "early_stopping_min_delta": self.early_stopping_min_delta,
            "restrict_synthetic_anomalies_to_target_mask": (
                self.restrict_synthetic_anomalies_to_target_mask
            ),
            "max_samples_per_epoch": self.max_samples_per_epoch,
        }

    def _training_state(self) -> dict[str, Any]:
        return {
            "adaptor": self.adaptor.state_dict(),
            "discriminator": self.discriminator.state_dict(),
        }

    def _load_training_state(self, state: dict[str, Any]) -> None:
        self.adaptor.load_state_dict(state["adaptor"])
        self.discriminator.load_state_dict(state["discriminator"])

    def _logits(self, images: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        features = self.features(images)
        adapted = self.adaptor(features)
        classification = adapted if self.adapt_classification_features else features
        return self.discriminator(adapted, classification)

    def _image_score_components(
        self, images: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        return self._logits(images)

    def fit(
        self, train_loader, *, device, validation_loader=None,
        work_dir=None, resume=False,
    ) -> SuperSimpleNet:
        device = torch.device(device)
        if not self.fixed_training_duration and validation_loader is None:
            raise ValueError(
                "SuperSimpleNet early stopping requires validation_loader"
            )
        self.to(device)
        segmentation_parameters, decision_parameters = self.discriminator.parameter_groups()
        optimizer = torch.optim.AdamW([
            {"params": self.adaptor.parameters(), "lr": self.adaptor_learning_rate},
            {"params": segmentation_parameters, "lr": self.segmentation_learning_rate,
             "weight_decay": 1e-5},
            {"params": decision_parameters, "lr": self.decision_learning_rate,
             "weight_decay": 1e-5},
        ])
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer,
            milestones=[int(self.epochs * 0.8), int(self.epochs * 0.9)],
            gamma=self.scheduler_gamma,
        )
        best_auc = -math.inf
        best_state = None
        best_epoch = None
        validation_history: list[dict[str, Any]] = []
        no_improvement_validations = 0
        stopped_early = False
        steps = 0
        start_epoch = 0
        completed_epoch = 0
        checkpoint_path = (
            None if work_dir is None
            else Path(work_dir) / "supersimplenet_training.ckpt"
        )
        if resume:
            if checkpoint_path is None or not checkpoint_path.is_file():
                raise FileNotFoundError(
                    "resume=true requires work_dir/supersimplenet_training.ckpt"
                )
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            if checkpoint.get("format") != "supersimplenet_training_v1":
                raise ValueError("Unsupported SuperSimpleNet training checkpoint")
            if checkpoint.get("model_config") != self.checkpoint_config():
                raise ValueError("SuperSimpleNet training checkpoint config mismatch")
            self._load_training_state(checkpoint["model"])
            optimizer.load_state_dict(checkpoint["optimizer"])
            scheduler.load_state_dict(checkpoint["scheduler"])
            start_epoch = int(checkpoint["epoch"])
            completed_epoch = start_epoch
            if not 0 <= start_epoch <= self.epochs:
                raise ValueError("Invalid epoch in SuperSimpleNet training checkpoint")
            steps = int(checkpoint.get("steps", 0))
            best_auc = float(checkpoint.get("best_auc", -math.inf))
            best_state = checkpoint.get("best_state")
            best_epoch = checkpoint.get("best_epoch")
            validation_history = list(checkpoint.get("validation_history", []))
            no_improvement_validations = int(
                checkpoint.get("no_improvement_validations", 0)
            )
            stopped_early = bool(checkpoint.get("stopped_early", False))
            _restore_training_rng_state(checkpoint, train_loader)
        remaining_epochs = (
            () if stopped_early else range(start_epoch + 1, self.epochs + 1)
        )
        epoch_progress = tqdm(
            remaining_epochs,
            initial=start_epoch,
            total=self.epochs,
            desc="SuperSimpleNet training",
            unit="epoch",
            dynamic_ncols=True,
            leave=True,
        )
        for epoch in epoch_progress:
            completed_epoch = epoch
            self.train(True)
            samples = 0
            produced = False
            for batch in train_loader:
                produced = True
                images = _clean_images(batch, device)
                with torch.no_grad():
                    features = self.features(images)
                adapted = self.adaptor(features)
                target_masks = None
                if self.restrict_synthetic_anomalies_to_target_mask:
                    if "target_mask" not in batch:
                        raise KeyError(
                            "restrict_synthetic_anomalies_to_target_mask=true "
                            "requires target_mask in every training batch"
                        )
                    target_masks = F.interpolate(
                        batch["target_mask"].to(
                            device, non_blocking=True, dtype=torch.float32
                        ),
                        size=adapted.shape[-2:],
                        mode="nearest",
                    )
                noisy_features, noisy_adapted, target_mask = self.anomaly_generator(
                    features, adapted, target_masks=target_masks
                )
                classification = (
                    noisy_adapted if self.adapt_classification_features
                    else noisy_features
                )
                anomaly_map, score = self.discriminator(noisy_adapted, classification)
                target_label = target_mask.flatten(1).amax(1)

                focal = _ssn_focal_loss(
                    torch.sigmoid(anomaly_map), target_mask, reduction=None
                )
                truncated = torch.zeros_like(anomaly_map)
                normal = target_mask == 0
                anomalous = target_mask > 0
                truncated[normal] = torch.clamp(anomaly_map[normal] + self.margin, min=0)
                truncated[anomalous] = torch.clamp(-anomaly_map[anomalous] + self.margin, min=0)
                good_loss = truncated[normal].mean() if normal.any() else 0.0
                bad_loss = truncated[anomalous].mean() if anomalous.any() else 0.0
                loss = good_loss + bad_loss + focal.mean()
                loss = loss + _ssn_focal_loss(torch.sigmoid(score), target_label)

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if self.gradient_clip:
                    torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
                optimizer.step()
                steps += 1
                samples += images.shape[0]
                if self.max_samples_per_epoch and samples >= self.max_samples_per_epoch:
                    break
            if not produced:
                raise ValueError("train_loader produced no images")
            epoch_progress.set_postfix(
                loss=f"{loss.detach().item():.4f}",
                refresh=True,
            )
            scheduler.step()
            if (
                not self.fixed_training_duration
                and (epoch % self.validation_interval == 0 or epoch == self.epochs)
            ):
                self.fitted.fill_(True)
                auc = _validation_auc(
                    self, validation_loader, device, self.validation_batches
                )
                validation_history.append({
                    "epoch": epoch,
                    "image_auroc": auc,
                })
                epoch_progress.set_postfix(
                    val_auc=f"{auc:.4f}",
                    refresh=True,
                )
                if not math.isnan(auc):
                    if auc > best_auc + self.early_stopping_min_delta:
                        best_auc = auc
                        best_epoch = epoch
                        best_state = copy.deepcopy(self._training_state())
                        no_improvement_validations = 0
                    else:
                        no_improvement_validations += 1
                    if (
                        self.early_stopping_patience is not None
                        and no_improvement_validations
                        >= self.early_stopping_patience
                    ):
                        stopped_early = True
            if (
                checkpoint_path is not None
                and (
                    epoch % self.checkpoint_interval == 0
                    or epoch == self.epochs
                    or stopped_early
                )
            ):
                _save_checkpoint_atomic({
                    "format": "supersimplenet_training_v1",
                    "model_config": self.checkpoint_config(),
                    "model": self._training_state(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "epoch": epoch,
                    "steps": steps,
                    "best_auc": best_auc,
                    "best_state": best_state,
                    "best_epoch": best_epoch,
                    "validation_history": validation_history,
                    "no_improvement_validations": no_improvement_validations,
                    "stopped_early": stopped_early,
                    **_capture_training_rng_state(train_loader),
                }, checkpoint_path)
            if stopped_early:
                break
        if best_state is not None:
            self._load_training_state(best_state)
        self.fitted.fill_(True)
        self.eval()
        self.fit_summary = {
            "epochs": self.epochs, "steps": steps,
            "epochs_completed": completed_epoch,
            "fixed_training_duration": self.fixed_training_duration,
            "selected_validation_image_auroc": (
                None if self.fixed_training_duration or best_state is None else best_auc
            ),
            "selected_epoch": best_epoch,
            "stopped_early": stopped_early,
            "no_improvement_validations": no_improvement_validations,
            "validation_history": validation_history,
            "restrict_synthetic_anomalies_to_target_mask": (
                self.restrict_synthetic_anomalies_to_target_mask
            ),
        }
        if checkpoint_path is not None:
            _save_checkpoint_atomic({
                "format": "supersimplenet_training_v1",
                "model_config": self.checkpoint_config(),
                "model": self._training_state(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": completed_epoch,
                "steps": steps,
                "best_auc": best_auc,
                "best_state": best_state,
                "best_epoch": best_epoch,
                "validation_history": validation_history,
                "no_improvement_validations": no_improvement_validations,
                "stopped_early": stopped_early,
                **_capture_training_rng_state(train_loader),
            }, checkpoint_path)
        return self

    @torch.no_grad()
    def predict_with_raw(self, images: torch.Tensor):
        if not self.is_fitted:
            raise RuntimeError("SuperSimpleNet must be fitted before predict")
        self.eval()
        native_raw_map, classification_raw_score = self._logits(images)
        if self.image_score_mode == "classification_head":
            raw_score = classification_raw_score
        else:
            flattened = native_raw_map.flatten(1)
            topk = min(
                flattened.shape[1],
                max(1, math.ceil(self.image_score_fraction * flattened.shape[1])),
            )
            raw_score = flattened.topk(topk, dim=1).values.mean(dim=1)
        raw_map = native_raw_map
        raw_map = F.interpolate(
            raw_map, images.shape[-2:], mode="bilinear", align_corners=False
        )
        if self.gaussian_sigma > 0:
            kernel = 2 * math.ceil(3 * self.gaussian_sigma) + 1
            raw_map = gaussian_blur(
                raw_map, [kernel, kernel], [self.gaussian_sigma, self.gaussian_sigma]
            )
        anomaly_map = torch.sigmoid(raw_map)
        anomaly_score = torch.sigmoid(raw_score)
        prediction = AnomalyPrediction(anomaly_score, anomaly_map)
        return prediction, raw_score, raw_map

    @torch.no_grad()
    def predict(self, images: torch.Tensor) -> AnomalyPrediction:
        return self.predict_with_raw(images)[0]



In [ ]:
from __future__ import annotations

# Authenticated Google Drive download with atomic write and size verification.
DRIVE_SCOPE = "https://www.googleapis.com/auth/drive.file"
DRIVE_FOLDER_MIME = "application/vnd.google-apps.folder"


def get_kaggle_secret(name: str, *, required: bool = False) -> str | None:
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(
            f"Missing required Kaggle secret {name!r}. Add the same Google Drive "
            "credentials used by the roadmap notebook."
        )
    return value


def build_drive_service():
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    credentials = Credentials(
        token=None,
        refresh_token=get_kaggle_secret("GDRIVE_REFRESH_TOKEN", required=True),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_kaggle_secret("GDRIVE_CLIENT_ID", required=True),
        client_secret=get_kaggle_secret("GDRIVE_CLIENT_SECRET", required=True),
        scopes=[DRIVE_SCOPE],
    )
    return build("drive", "v3", credentials=credentials, cache_discovery=False)


def parse_drive_item_id(value: str) -> str:
    from urllib.parse import parse_qs, urlparse

    value = str(value or "").strip()
    if not value:
        raise ValueError("Google Drive ID/URL cannot be empty")
    if "/" not in value and "?" not in value:
        item_id = value
    else:
        parsed = urlparse(value)
        parts = [part for part in parsed.path.split("/") if part]
        item_id = parse_qs(parsed.query).get("id", [None])[0]
        if item_id is None:
            for marker in ("d", "folders"):
                if marker in parts and parts.index(marker) + 1 < len(parts):
                    item_id = parts[parts.index(marker) + 1]
                    break
    if not item_id or not all(
        character.isalnum() or character in "-_" for character in item_id
    ):
        raise ValueError(f"Invalid Google Drive ID/URL: {value!r}")
    return item_id


def drive_children(service, parent_id: str, name: str | None = None) -> list[dict]:
    backslash = chr(92)
    escaped_parent = parent_id.replace("'", backslash + "'")
    query = [f"'{escaped_parent}' in parents", "trashed = false"]
    if name is not None:
        escaped_name = name.replace(backslash, backslash * 2).replace(
            "'", backslash + "'"
        )
        query.append(f"name = '{escaped_name}'")
    return service.files().list(
        q=" and ".join(query),
        spaces="drive",
        fields="files(id,name,mimeType,size,parents)",
        pageSize=1000,
    ).execute(num_retries=3).get("files", [])


def selected_drive_run() -> dict[str, Any]:
    if DRIVE_RUN_FOLDER_ID_OVERRIDE:
        return {
            "model": MODEL_NAME,
            "folder_id": parse_drive_item_id(DRIVE_RUN_FOLDER_ID_OVERRIDE),
            "checkpoint_file_id": None,
            "compatible_with_current_preset": None,
            "comment": "Manual override; strict checkpoint configuration still applies.",
        }
    if SELECTED_DRIVE_RUN is None:
        raise RuntimeError(
            f"No verified Drive checkpoint is registered for {MODEL_NAME}. "
            "Use CHECKPOINT_SOURCE='local' and set MODEL_CHECKPOINT_PATH."
        )
    return dict(SELECTED_DRIVE_RUN)


def resolve_drive_checkpoint_file(service, run: Mapping[str, Any]) -> dict[str, Any]:
    folder_id = parse_drive_item_id(run["folder_id"])
    matches = [
        item
        for item in drive_children(service, folder_id, "model.ckpt")
        if item["mimeType"] != DRIVE_FOLDER_MIME
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one model.ckpt in Drive folder {folder_id}, "
            f"found {len(matches)}"
        )
    item = matches[0]
    expected_file_id = run.get("checkpoint_file_id")
    if expected_file_id and item["id"] != expected_file_id:
        raise ValueError(
            "Drive folder checkpoint ID changed: "
            f"expected {expected_file_id}, found {item['id']}"
        )
    return item


def download_drive_checkpoint(
    service,
    run: Mapping[str, Any],
) -> tuple[Path, dict[str, Any]]:
    from googleapiclient.http import MediaIoBaseDownload

    item = resolve_drive_checkpoint_file(service, run)
    file_id = item["id"]
    destination = DRIVE_CHECKPOINT_CACHE_DIR / MODEL_NAME / f"{file_id}.ckpt"
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(item.get("size") or 0)
    if (
        not FORCE_CHECKPOINT_DOWNLOAD
        and destination.is_file()
        and expected_size
        and destination.stat().st_size == expected_size
    ):
        print(f"Reusing size-verified cached checkpoint: {destination}")
        return destination, {
            **item,
            "cached": True,
            "run_folder_id": parse_drive_item_id(run["folder_id"]),
        }

    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    request = service.files().get_media(fileId=file_id)
    try:
        with temporary.open("wb") as handle, tqdm(
            total=expected_size or None,
            desc=f"Downloading {MODEL_NAME} model.ckpt",
            unit="B",
            unit_scale=True,
            leave=False,
            dynamic_ncols=True,
        ) as progress:
            downloader = MediaIoBaseDownload(
                handle,
                request,
                chunksize=8 * 1024 * 1024,
            )
            downloaded = 0
            done = False
            while not done:
                status, done = downloader.next_chunk(num_retries=5)
                if status is not None:
                    current = int(status.resumable_progress)
                    progress.update(max(0, current - downloaded))
                    downloaded = current
        if expected_size and temporary.stat().st_size != expected_size:
            raise IOError(
                f"Incomplete Drive checkpoint: {temporary.stat().st_size} / "
                f"{expected_size} bytes"
            )
        os.replace(temporary, destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return destination, {
        **item,
        "cached": False,
        "run_folder_id": parse_drive_item_id(run["folder_id"]),
    }


def download_selected_checkpoint_from_drive() -> tuple[Path, dict[str, Any]]:
    run = selected_drive_run()
    if run.get("model") != MODEL_NAME:
        raise ValueError(
            f"Drive run model {run.get('model')!r} does not match {MODEL_NAME!r}"
        )
    service = build_drive_service()
    checkpoint_path, metadata = download_drive_checkpoint(service, run)
    print(
        f"Drive run folder: {parse_drive_item_id(run['folder_id'])} | "
        f"checkpoint file: {metadata['id']} | bytes={metadata.get('size', 'unknown')}"
    )
    return checkpoint_path, {"run": run, "file": metadata}


In [ ]:
from __future__ import annotations

def load_frozen_supersimplenet(
    checkpoint_path: str | Path, *, device: torch.device | str,
) -> tuple[SuperSimpleNet, dict[str, Any]]:
    """Restore the selected saved model exactly, without inference overrides."""
    checkpoint_path = Path(checkpoint_path).expanduser()
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f"Final model checkpoint is missing: {checkpoint_path}")
    # Trusted project checkpoints only; do not fall back to unsafe pickle loading.
    payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    required = {"schema_version", "model_class", "model_config", "model_state_dict"}
    if not isinstance(payload, dict) or not required <= set(payload):
        raise ValueError("Expected a final model.ckpt, not a training checkpoint")
    if payload["schema_version"] != CHECKPOINT_SCHEMA_VERSION:
        raise ValueError("Unsupported final checkpoint schema")
    if str(payload["model_class"]).rsplit(".", 1)[-1] != "SuperSimpleNet":
        raise ValueError("The supplied checkpoint is not SuperSimpleNet")
    if not isinstance(payload["model_config"], dict):
        raise ValueError("Checkpoint model_config must be a dictionary")
    original_saved_config = dict(payload["model_config"])
    saved_config = dict(original_saved_config)
    canonicalized_sequence_fields = []
    for key in ("layers", "input_size"):
        if isinstance(saved_config.get(key), tuple):
            saved_config[key] = list(saved_config[key])
            canonicalized_sequence_fields.append(key)
    for key, expected in EXPECTED_CHECKPOINT_SETTINGS.items():
        if key not in saved_config or saved_config[key] != expected:
            raise ValueError(
                f"The selected SuperSimpleNet checkpoint requires {key}={expected!r}; "
                f"saved={saved_config.get(key)!r}. No scoring or model-selection override is applied."
            )

    metadata = dict(payload.get("metadata") or {})
    metadata_model = dict(metadata.get("model") or {})
    if metadata_model.get("name") is not None and metadata_model["name"] != MODEL_NAME:
        raise ValueError("Checkpoint model metadata does not match SuperSimpleNet")
    preprocessing = dict(metadata.get("preprocessing") or {})
    expected_preprocessing = {
        "input_size": list(MODEL_INPUT_SIZE),
        "resize": list(MODEL_INPUT_SIZE),
        "resize_shorter_side": None,
        "center_crop": None,
        "pose_crop_enabled": True,
        "normalize_mean": list(IMAGENET_MEAN),
        "normalize_std": list(IMAGENET_STD),
    }
    for key, expected in expected_preprocessing.items():
        if key in preprocessing:
            value = preprocessing[key]
            if isinstance(expected, list) and isinstance(value, tuple):
                value = list(value)
            if value != expected:
                raise ValueError(f"Checkpoint preprocessing mismatch: {key}")
    saved_pose_crop = (preprocessing.get("pose_crops") or {}).get("A_overhead")
    if saved_pose_crop is not None and tuple(saved_pose_crop) != tuple(POSE_CROPS["A_overhead"]):
        raise ValueError("Checkpoint pose-A crop does not match the fixed crop720")
    dataset_metadata = dict(metadata.get("dataset") or {})
    if dataset_metadata.get("camera_poses") is not None:
        if tuple(dataset_metadata["camera_poses"]) != tuple(CAMERA_POSES):
            raise ValueError("The selected checkpoint must have been trained on pose A only")

    # Saved settings, not current preset defaults, determine the restored model.
    constructor_fields = set(inspect.signature(SuperSimpleNet.__init__).parameters) - {"self"}
    unknown_fields = set(saved_config) - constructor_fields
    if unknown_fields:
        raise ValueError(f"Unrecognized SuperSimpleNet checkpoint fields: {sorted(unknown_fields)}")
    constructor_config = dict(saved_config)
    saved_pretrained = constructor_config["pretrained"]
    constructor_config["pretrained"] = False
    restored = SuperSimpleNet(**constructor_config)
    restored.pretrained = saved_pretrained
    if restored.checkpoint_config() != saved_config:
        raise ValueError(
            "Saved SuperSimpleNet configuration does not match the current implementation exactly"
        )
    restored._prepare_state_dict_for_load(payload["model_state_dict"])
    restored.load_state_dict(payload["model_state_dict"], strict=True)
    if not restored.is_fitted:
        raise ValueError("The supplied final checkpoint contains an unfitted model")
    restored.fit_summary = dict(metadata.get("fit_summary") or {})
    restored.to(device).eval()
    restore_info = {
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_size_bytes": checkpoint_path.stat().st_size,
        "checkpoint_model_class": payload["model_class"],
        "source_run_id": metadata.get("run_id"),
        "saved_model_config": original_saved_config,
        "inference_model_config": restored.checkpoint_config(),
        "canonicalized_sequence_fields": canonicalized_sequence_fields,
        "saved_gaussian_sigma": float(saved_config["gaussian_sigma"]),
        "inference_gaussian_sigma": float(restored.gaussian_sigma),
        "image_score_mode": restored.image_score_mode,
        "image_score_fraction": restored.image_score_fraction,
        "source_manifest_sha256": dataset_metadata.get("manifest_sha256"),
        "training_performed": False,
        "calibration_performed": False,
        "accuracy_evaluation_performed": False,
        "score_selection_performed": False,
        "sigma_selection_performed": False,
        "inference_autocast": False,
    }
    return restored, restore_info


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_json_atomic(value: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)
    return path


In [ ]:
from __future__ import annotations

import threading

def _latency_summary(milliseconds: list[float]) -> dict[str, float]:
    if not milliseconds:
        raise ValueError("Latency samples cannot be empty")
    ordered = sorted(float(value) for value in milliseconds)

    def percentile(fraction: float) -> float:
        position = fraction * (len(ordered) - 1)
        lower, upper = math.floor(position), math.ceil(position)
        weight = position - lower
        return ordered[lower] * (1 - weight) + ordered[upper] * weight

    return {
        "minimum_ms": ordered[0],
        "mean_ms": sum(ordered) / len(ordered),
        "median_ms": percentile(0.50),
        "p90_ms": percentile(0.90),
        "p95_ms": percentile(0.95),
        "maximum_ms": ordered[-1],
    }


def _cuda_memory_baseline(device: torch.device) -> tuple[int, int] | None:
    if device.type != "cuda":
        return None
    torch.cuda.synchronize(device)
    torch.cuda.empty_cache()
    baseline = (
        torch.cuda.memory_allocated(device), torch.cuda.memory_reserved(device)
    )
    torch.cuda.reset_peak_memory_stats(device)
    return baseline


def _cuda_memory_report(
    device: torch.device, baseline: tuple[int, int] | None,
) -> dict[str, float] | None:
    if baseline is None:
        return None
    allocated, reserved = baseline
    peak_allocated = torch.cuda.max_memory_allocated(device)
    peak_reserved = torch.cuda.max_memory_reserved(device)
    return {
        "baseline_allocated_mib": allocated / 2**20,
        "baseline_reserved_mib": reserved / 2**20,
        "peak_allocated_mib": peak_allocated / 2**20,
        "peak_reserved_mib": peak_reserved / 2**20,
        "incremental_peak_allocated_mib": (peak_allocated - allocated) / 2**20,
        "incremental_peak_reserved_mib": (peak_reserved - reserved) / 2**20,
    }


@torch.inference_mode()
def benchmark_anomaly_inference(
    detector: Any,
    loader: Any,
    *,
    device: torch.device | str,
    warmup_runs: int = 10,
    timed_runs: int = 50,
    split: str = "validation",
) -> dict[str, Any]:
    """Measure predict-only and sequential batch-one pipeline latency.

    ``predict_only`` reuses one preprocessed, device-resident input. The other
    measurements include the loader's complete work, which for the synthetic
    dataset includes annotation I/O and aligned mask transforms. They are not
    a mask-free deployment benchmark. CUDA allocator peaks include all live
    tensors on the selected device, not just the model, and exclude driver
    allocations. No power sensor or energy estimate is used.
    """
    if getattr(loader, "batch_size", None) != 1:
        raise ValueError("Inference benchmark loader must use batch_size=1")
    if getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Sequential inference benchmark requires num_workers=0")
    if not isinstance(warmup_runs, int) or not isinstance(timed_runs, int):
        raise TypeError("warmup_runs and timed_runs must be integers")
    if warmup_runs < 1 or timed_runs < 1:
        raise ValueError("warmup_runs and timed_runs must be positive")
    if not detector.is_fitted:
        raise RuntimeError("Fit or restore the detector before benchmarking")

    device = torch.device(device)
    uses_cuda = device.type == "cuda"
    was_training = detector.training
    detector.to(device)
    detector.eval()

    def synchronize() -> None:
        if uses_cuda:
            torch.cuda.synchronize(device)

    def next_batch(iterator):
        try:
            return next(iterator), iterator
        except StopIteration:
            iterator = iter(loader)
            try:
                return next(iterator), iterator
            except StopIteration as error:
                raise ValueError("Inference benchmark loader is empty") from error

    try:
        iterator = iter(loader)
        batch, iterator = next_batch(iterator)
        images = batch["image"].to(device, non_blocking=uses_cuda)
        if images.ndim != 4 or images.shape[0] != 1:
            raise ValueError("Inference benchmark requires one BCHW image")
        input_shape, input_dtype = list(images.shape), str(images.dtype)
        for _ in range(warmup_runs):
            prediction = detector.predict(images)
            del prediction
        synchronize()
        baseline = _cuda_memory_baseline(device)
        predict_ms = []
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            prediction = detector.predict(images)
            synchronize()
            predict_ms.append((time.perf_counter() - start) * 1_000)
            # Do not retain the previous output across the next forward.
            del prediction
        predict_memory = _cuda_memory_report(device, baseline)
        del images, batch, iterator

        iterator = iter(loader)
        for _ in range(warmup_runs):
            batch, iterator = next_batch(iterator)
            images = batch["image"].to(device, non_blocking=uses_cuda)
            prediction = detector.predict(images)
            synchronize()
            del prediction, images, batch
        baseline = _cuda_memory_baseline(device)
        preprocessing_ms, accelerator_ms, pipeline_ms = [], [], []
        iterator = iter(loader)
        for _ in range(timed_runs):
            synchronize()
            start = time.perf_counter()
            batch, iterator = next_batch(iterator)
            preprocessing_end = time.perf_counter()
            images = batch["image"].to(device, non_blocking=uses_cuda)
            if list(images.shape) != input_shape or str(images.dtype) != input_dtype:
                raise ValueError("Benchmark inputs must have one stable shape and dtype")
            prediction = detector.predict(images)
            synchronize()
            end = time.perf_counter()
            preprocessing_ms.append((preprocessing_end - start) * 1_000)
            accelerator_ms.append((end - preprocessing_end) * 1_000)
            pipeline_ms.append((end - start) * 1_000)
            del prediction, images, batch
        pipeline_memory = _cuda_memory_report(device, baseline)

        predict_summary = _latency_summary(predict_ms)
        accelerator_summary = _latency_summary(accelerator_ms)
        pipeline_summary = _latency_summary(pipeline_ms)
        parameters = list(detector.parameters())
        buffers = list(detector.buffers())
        bank = getattr(detector, "memory_bank", None)
        return {
            "schema_version": 1,
            "scope": (
                "predict_only: one device-resident preprocessed image; pipeline: "
                "sequential loader (including annotation I/O and mask transforms), "
                "host-to-device transfer and detector.predict; no worker prefetch"
            ),
            "memory_scope": (
                "CUDA allocator totals include the model, memory bank and other live "
                "tensors on this device; driver allocations are excluded"
            ),
            "split": split,
            "batch_size": 1,
            "warmup_runs": warmup_runs,
            "timed_runs": timed_runs,
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(device) if uses_cuda else None,
            "input_shape": input_shape,
            "input_dtype": input_dtype,
            "model": type(detector).__name__,
            "gaussian_sigma": getattr(detector, "gaussian_sigma", None),
            "total_parameters": sum(parameter.numel() for parameter in parameters),
            "trainable_parameters": sum(
                parameter.numel() for parameter in parameters if parameter.requires_grad
            ),
            "parameters_mib": sum(
                parameter.numel() * parameter.element_size() for parameter in parameters
            ) / 2**20,
            "buffers_mib": sum(
                buffer.numel() * buffer.element_size() for buffer in buffers
            ) / 2**20,
            "memory_bank_shape": list(bank.shape) if bank is not None else None,
            "memory_bank_mib": (
                bank.numel() * bank.element_size() / 2**20 if bank is not None else None
            ),
            "latency": {
                "predict_only": predict_summary,
                "decode_and_preprocess": _latency_summary(preprocessing_ms),
                "host_to_device_and_predict": accelerator_summary,
                "end_to_end": pipeline_summary,
            },
            "throughput_images_per_second": 1_000 / accelerator_summary["mean_ms"],
            "predict_only_images_per_second": 1_000 / predict_summary["mean_ms"],
            "end_to_end_images_per_second": 1_000 / pipeline_summary["mean_ms"],
            "cuda_memory": pipeline_memory,
            "predict_only_cuda_memory": predict_memory,
            "environment": {
                "python_version": sys.version.split()[0],
                "platform": platform.platform(),
                "torch_version": str(torch.__version__),
                "cuda_version": torch.version.cuda,
                "cudnn_version": torch.backends.cudnn.version(),
                "cudnn_benchmark": torch.backends.cudnn.benchmark,
                "cudnn_deterministic": torch.backends.cudnn.deterministic,
                "matmul_allow_tf32": torch.backends.cuda.matmul.allow_tf32,
                "cudnn_allow_tf32": torch.backends.cudnn.allow_tf32,
                "float32_matmul_precision": torch.get_float32_matmul_precision(),
            },
            "energy_joules_per_image": None,
            "energy_note": "Not measured; latency alone is not an energy measurement",
        }
    finally:
        detector.train(was_training)

def _nvml_handle_for_cuda(nvml: Any, device: torch.device) -> tuple[Any, str]:
    """Match by CUDA UUID; never assume CUDA and NVML ordinals are identical."""
    properties = torch.cuda.get_device_properties(device)
    cuda_uuid = getattr(properties, "uuid", None)
    if cuda_uuid:
        uuid_text = cuda_uuid.decode("ascii") if isinstance(cuda_uuid, bytes) else str(cuda_uuid)
        if not uuid_text.startswith(("GPU-", "MIG-")):
            uuid_text = "GPU-" + uuid_text
        try:
            return nvml.nvmlDeviceGetHandleByUUID(uuid_text), "cuda_uuid"
        except Exception as error:
            raise RuntimeError(f"Cannot match the CUDA UUID to NVML: {error}") from error
    if nvml.nvmlDeviceGetCount() == 1:
        return nvml.nvmlDeviceGetHandleByIndex(0), "single_nvml_device"
    raise RuntimeError(
        "CUDA UUID is unavailable and multiple NVML GPUs exist; "
        "refusing to measure an ambiguous physical device"
    )


def _nvml_counter_energy_window(
    predict_once: Any, nvml: Any, handle: Any, duration_seconds: float,
) -> dict[str, Any]:
    """Counter units are mJ; inference failures propagate rather than being hidden."""
    method = "nvml_total_energy_counter"
    try:
        before = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
        if before < 0:
            raise ValueError("negative energy counter")
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    count = 0
    while count == 0 or time.perf_counter() - start < duration_seconds:
        predict_once()
        count += 1
    try:
        after = int(nvml.nvmlDeviceGetTotalEnergyConsumption(handle))
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    elapsed = time.perf_counter() - start
    if after <= before:
        return {
            "status": "unavailable", "method": method,
            "reason": "Energy counter did not advance or was reset during measurement",
        }
    joules = (after - before) / 1_000.0
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "counter_start_millijoules": before,
        "counter_end_millijoules": after,
        "power_samples": [],
    }


def _nvml_power_energy_window(
    predict_once: Any, nvml: Any, handle: Any,
    duration_seconds: float, sample_interval_seconds: float,
) -> dict[str, Any]:
    """Integrate measured GPU power, with explicit boundary samples and no TDP estimate."""
    method = "nvml_power_integration"

    def read_watts() -> float:
        watts = float(nvml.nvmlDeviceGetPowerUsage(handle)) / 1_000.0
        if not math.isfinite(watts) or watts <= 0:
            raise ValueError("NVML power reading must be positive and finite")
        return watts

    try:
        first_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    start = time.perf_counter()
    samples = [(start, first_watts)]
    errors = []
    stop = threading.Event()

    def sample_power() -> None:
        while not stop.wait(sample_interval_seconds):
            try:
                watts = read_watts()
                samples.append((time.perf_counter(), watts))
            except Exception as error:
                errors.append(str(error))
                return

    worker = threading.Thread(target=sample_power, daemon=True)
    worker.start()
    count = 0
    try:
        while count == 0 or time.perf_counter() - start < duration_seconds:
            predict_once()
            count += 1
            if errors:
                break
    finally:
        stop.set()
        worker.join()
    if errors:
        return {"status": "unavailable", "method": method, "reason": errors[0]}
    try:
        final_watts = read_watts()
    except Exception as error:
        return {"status": "unavailable", "method": method, "reason": str(error)}
    samples.append((time.perf_counter(), final_watts))
    elapsed = samples[-1][0] - start
    # Two endpoints alone are not adequate for the sampled-power fallback.
    if len(samples) < 3:
        return {
            "status": "unavailable", "method": method,
            "reason": "Too few power samples; increase the measurement window",
        }
    joules = sum(
        (right[0] - left[0]) * (left[1] + right[1]) * 0.5
        for left, right in zip(samples, samples[1:])
    )
    return {
        "status": "measured", "method": method,
        "inferences": count, "duration_seconds": elapsed,
        "gpu_energy_joules": joules, "energy_joules_per_image": joules / count,
        "average_gpu_power_watts": joules / elapsed,
        "power_sample_interval_seconds_requested": sample_interval_seconds,
        "power_sample_count": len(samples),
        "power_samples": [
            {"elapsed_seconds": timestamp - start, "power_watts": watts}
            for timestamp, watts in samples
        ],
    }


@torch.inference_mode()
def measure_anomaly_inference_energy(
    detector: Any, loader: Any, *, device: torch.device | str,
    repeats: int = 3, duration_seconds: float = 30.0,
    warmup_seconds: float = 5.0, sample_interval_seconds: float = 0.1,
    method: str = "auto",
) -> dict[str, Any]:
    """Measure device-wide GPU J/image in a separate sustained predict-only phase.

    The model and one preprocessed input are already GPU resident. Each complete
    predict is synchronized, and its output discarded. Includes GPU idle/boundary
    energy, excludes CPU, I/O and host-to-device energy. Raw total energy is not
    idle-subtracted and is not attributed to an individual process. Sensor errors
    yield an explicit unavailable/partial report; prediction errors propagate.
    """
    if not isinstance(repeats, int) or isinstance(repeats, bool) or repeats < 1:
        raise ValueError("Energy repeats must be a positive integer")
    for name, value in (
        ("duration_seconds", duration_seconds),
        ("sample_interval_seconds", sample_interval_seconds),
    ):
        if not math.isfinite(value) or value <= 0:
            raise ValueError(f"{name} must be positive and finite")
    if not math.isfinite(warmup_seconds) or warmup_seconds < 0:
        raise ValueError("warmup_seconds must be non-negative and finite")
    if method not in {"auto", "counter", "power"}:
        raise ValueError("Energy method must be auto, counter or power")
    if getattr(loader, "batch_size", None) != 1 or getattr(loader, "num_workers", 0) != 0:
        raise ValueError("Energy requires batch_size=1 and num_workers=0")
    if not detector.is_fitted:
        raise RuntimeError("Restore the detector before measuring energy")
    device = torch.device(device)
    report = {
        "schema_version": 1,
        "status": "unavailable", "reason": None, "method": None,
        "metric": "device_wide_gpu_energy_per_inference",
        "scope": (
            "predict-only repeated synchronized inference; model and one preprocessed "
            "image are CUDA resident; excludes input transfer, CPU and loader work"
        ),
        "device": str(device), "batch_size": 1,
        "repeats_requested": repeats,
        "duration_seconds_per_repeat_requested": duration_seconds,
        "warmup_seconds_requested": warmup_seconds,
        "sample_interval_seconds_requested": sample_interval_seconds,
        "method_requested": method,
        "idle_subtracted": False, "process_attributed": False,
        "energy_joules_per_image": None,
        "average_gpu_power_watts": None,
        "energy_joules_per_image_std": None,
        "total_inferences": 0, "runs": [], "warnings": [],
    }
    if device.type != "cuda" or not torch.cuda.is_available():
        report["reason"] = "CUDA GPU is required for NVML energy measurements"
        return report
    try:
        import pynvml
    except ImportError:
        report["reason"] = "Install nvidia-ml-py (import pynvml) to read GPU energy/power"
        return report
    try:
        pynvml.nvmlInit()
    except Exception as error:
        report["reason"] = f"NVML initialization failed: {error}"
        return report

    was_training = detector.training
    try:
        try:
            handle, mapping = _nvml_handle_for_cuda(pynvml, device)
            report["device_mapping"] = mapping
            uuid = pynvml.nvmlDeviceGetUUID(handle)
            report["nvml_gpu_uuid"] = uuid.decode() if isinstance(uuid, bytes) else str(uuid)
            name = pynvml.nvmlDeviceGetName(handle)
            report["gpu_name"] = name.decode() if isinstance(name, bytes) else str(name)
        except Exception as error:
            report["reason"] = f"NVML device mapping failed: {error}"
            return report
        try:
            processes = pynvml.nvmlDeviceGetComputeRunningProcesses(handle)
            report["gpu_compute_process_count_at_start"] = len(processes)
            if len(processes) > 1:
                report["warnings"].append(
                    "Multiple compute processes are visible on this GPU; "
                    "device-wide energy cannot be assigned exclusively to this model"
                )
        except Exception:
            report["gpu_compute_process_count_at_start"] = None
        try:
            report["power_limit_watts"] = pynvml.nvmlDeviceGetPowerManagementLimit(handle) / 1_000.0
        except Exception:
            report["power_limit_watts"] = None
        detector.to(device).eval()
        try:
            batch = next(iter(loader))
        except StopIteration as error:
            raise ValueError("Energy measurement loader is empty") from error
        images = batch["image"].to(device)
        del batch
        if images.ndim != 4 or images.shape[0] != 1 or images.dtype != torch.float32:
            raise ValueError("Energy comparison requires one FP32 BCHW image")
        report["input_shape"] = list(images.shape)
        report["input_dtype"] = str(images.dtype)

        def predict_once() -> None:
            prediction = detector.predict(images)
            torch.cuda.synchronize(device)
            del prediction

        torch.cuda.synchronize(device)
        warmup_start = time.perf_counter()
        predict_once()
        while time.perf_counter() - warmup_start < warmup_seconds:
            predict_once()
        report["warmup_seconds_actual"] = time.perf_counter() - warmup_start
        chosen_method = "power" if method == "power" else "counter"
        for repeat in range(repeats):
            if chosen_method == "counter":
                run = _nvml_counter_energy_window(
                    predict_once, pynvml, handle, duration_seconds
                )
                # Auto chooses one successful backend on the first repeat only;
                # never average counter and integrated-power repeats together.
                if repeat == 0 and method == "auto" and run["status"] != "measured":
                    report["warnings"].append("Counter unavailable: " + str(run.get("reason")))
                    chosen_method = "power"
                    run = _nvml_power_energy_window(
                        predict_once, pynvml, handle,
                        duration_seconds, sample_interval_seconds,
                    )
            else:
                run = _nvml_power_energy_window(
                    predict_once, pynvml, handle,
                    duration_seconds, sample_interval_seconds,
                )
            run["repeat"] = repeat + 1
            report["runs"].append(run)
            if run["status"] != "measured":
                report["status"] = "partial" if repeat else "unavailable"
                report["reason"] = run.get("reason", "GPU energy sensor failed")
                return report
        runs = report["runs"]
        total_images = sum(run["inferences"] for run in runs)
        total_joules = sum(run["gpu_energy_joules"] for run in runs)
        total_seconds = sum(run["duration_seconds"] for run in runs)
        values = [run["energy_joules_per_image"] for run in runs]
        mean = sum(values) / len(values)
        report.update({
            "status": "measured", "reason": None,
            "method": runs[0]["method"],
            "total_inferences": total_images,
            "total_gpu_energy_joules": total_joules,
            "total_duration_seconds": total_seconds,
            "energy_joules_per_image": total_joules / total_images,
            "average_gpu_power_watts": total_joules / total_seconds,
            "energy_joules_per_image_std": (
                math.sqrt(sum((value - mean) ** 2 for value in values) / (len(values) - 1))
                if len(values) > 1 else 0.0
            ),
        })
        return report
    finally:
        detector.train(was_training)
        pynvml.nvmlShutdown()


## Measurement protocol and validity checks

Use a fresh session with only this detector resident on the GPU. The frozen WideResNet-50-2 backbone, feature adaptor, full discriminator and output postprocessing run in FP32. The decision branch still executes, as in the authoritative implementation, even though the selected image score is the top-1% mean on the native anomaly map; σ4 map smoothing is included in every `predict` call. No feature cache, autocast, compilation, quantization or synthetic-input shortcut is used.

The benchmark matches the dedicated PatchCore protocol: batch 1, 10 warm-up iterations and 50 timed iterations **per scope**, CUDA synchronization, sequential loader, no worker prefetch and no pinned-memory shortcut.

- **Predict-only:** the same preprocessed image is already on the device. This excludes loading, transforms and host-to-device transfer.
- **Host-to-device + predict:** one image is transferred after each loader iteration.
- **End-to-end:** the existing synthetic loader also reads and transforms target/anomaly masks. This is an evaluation pipeline, **not** a mask-free deployment measurement.
- Report CUDA allocated/reserved baseline, total peak and incremental peak separately. These are PyTorch allocator measurements, not process/driver memory.
- Latencies from the separate phases cannot be subtracted to infer transfer cost. Compare models on the same GPU and the same named scope.

The checkpoint's saved architecture, image-scoring mode, σ4 smoothing and preprocessing are checked before restore. A different training variant (for example, wheel-restricted synthetic anomalies) is rejected; no checkpoint default or inference override is silently substituted. A manifest mismatch or unexpected validation count is recorded and warned about, not filtered: these do not affect the fixed input shape. There is no full-dataset mask audit or visibility gate. Energy is measured in the separate phase below; latency alone is not an energy estimate.

### GPU energy (separate from the latency windows)

The default is three **30 s** sustained predict-only windows after a **5 s**
warm-up, batch 1, FP32, and the same preprocessed image already on the GPU.
The complete predict call and its postprocessing run every iteration; outputs
are discarded. No feature cache, refit, recalibration or input shortcut is used.

NVML's total-energy counter is preferred (mJ converted to J). When unavailable
or non-advancing, the first window chooses the measured-power fallback:
100 ms requested sampling and trapezoidal integration. A single backend is
used for all successful repeats; a partial sensor failure leaves the aggregate
null and records the reason. Inference errors still propagate.

The result is **device-wide GPU J/image**, including GPU idle power and memory,
not per-process or whole-system energy. CPU, loader and input-transfer work are
excluded; idle power is not subtracted. The CUDA/NVML devices are matched by
UUID, and additional visible compute processes are flagged. Use a fresh T4
session with no other model resident for the comparison. Actual power and
sensor readings are saved, never estimated from nominal GPU power or latency.

The energy phase adds about 95 seconds when the counter succeeds. Read the
**final saved JSON or the post-energy printout**, not the earlier latency
printout whose energy field may still be null.


## Execute the benchmark

Run the cells below in order. Only the final checkpoint and the validation loader are required. Training and test loaders are not built; no accuracy metrics or model selection are recomputed.


In [ ]:
from __future__ import annotations

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError("Select a Kaggle GPU session, preferably Tesla T4 to match the other benchmarks")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("highest")
if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(DEVICE)
    print(f"GPU: {gpu_name}")
    if "T4" not in gpu_name:
        print("WARNING: this is not a T4; do not compare these latencies directly with T4 results")

if CHECKPOINT_SOURCE == "google_drive":
    checkpoint_path, download_info = download_selected_checkpoint_from_drive()
else:
    if MODEL_CHECKPOINT_PATH is None:
        raise ValueError("Set MODEL_CHECKPOINT_PATH to your trusted final SuperSimpleNet model.ckpt")
    checkpoint_path = Path(MODEL_CHECKPOINT_PATH).expanduser()
    download_info = {"source": "local", "path": str(checkpoint_path)}
detector, restore_info = load_frozen_supersimplenet(checkpoint_path, device=DEVICE)
print({
    "source_run_id": restore_info["source_run_id"],
    "input_size": detector.input_size,
    "image_score_mode": detector.image_score_mode,
    "image_score_fraction": detector.image_score_fraction,
    "saved_sigma": restore_info["saved_gaussian_sigma"],
    "inference_sigma": detector.gaussian_sigma,
    "checkpoint_mib": restore_info["checkpoint_size_bytes"] / 2**20,
    "training_performed": False,
})


In [ ]:
from __future__ import annotations

DATASET_ROOT = (
    Path(DATASET_ROOT_OVERRIDE).expanduser().resolve()
    if DATASET_ROOT_OVERRIDE is not None else find_dataset_root()
)
preprocessor = WheelPreprocessor(PreprocessingConfig(
    resize=MODEL_INPUT_SIZE,
    resize_shorter_side=None,
    center_crop=None,
    pose_crop_enabled=True,
    pose_crops=POSE_CROPS,
    normalize_mean=IMAGENET_MEAN,
    normalize_std=IMAGENET_STD,
    augmentations_enabled=False,
))
benchmark_loader = build_dataloader(
    DATASET_ROOT, BENCHMARK_SPLIT,
    batch_size=BENCHMARK_BATCH_SIZE,
    num_workers=BENCHMARK_NUM_WORKERS,
    pin_memory=BENCHMARK_PIN_MEMORY,
    seed=RANDOM_SEED,
    preprocessing=preprocessor,
    camera_poses=CAMERA_POSES,
)
condition_counts = Counter(row["condition"] for row in benchmark_loader.dataset.rows)
if condition_counts != Counter({"clean": 296, "hole": 96}):
    print(f"WARNING: validation counts differ from the original pose-A dataset: {dict(condition_counts)}")
manifest_sha256 = sha256_file(DATASET_ROOT / "samples.csv")
source_manifest = restore_info["source_manifest_sha256"]
dataset_matches_source_manifest = None if source_manifest is None else manifest_sha256 == source_manifest
if dataset_matches_source_manifest is False:
    print("WARNING: attached samples.csv differs from the checkpoint dataset manifest; this will be recorded")
print({
    "dataset_root": str(DATASET_ROOT),
    "split": BENCHMARK_SPLIT,
    "camera_poses": CAMERA_POSES,
    "condition_counts": dict(condition_counts),
    "input_size": MODEL_INPUT_SIZE,
    "pose_crop": POSE_CROPS["A_overhead"],
    "manifest_sha256": manifest_sha256,
    "dataset_matches_source_manifest": dataset_matches_source_manifest,
})


In [ ]:
from __future__ import annotations

print("Benchmarking SuperSimpleNet inference only; no fit or accuracy evaluation")
# Outputs of predict() are discarded inside the benchmark to avoid inflated peaks.
benchmark_report = benchmark_anomaly_inference(
    detector, benchmark_loader, device=DEVICE,
    warmup_runs=BENCHMARK_WARMUP_RUNS,
    timed_runs=BENCHMARK_TIMED_RUNS,
    split=BENCHMARK_SPLIT,
)
if benchmark_report["input_shape"] != [1, 3, *MODEL_INPUT_SIZE]:
    raise ValueError("Unexpected benchmark input shape")
if benchmark_report["input_dtype"] != "torch.float32":
    raise ValueError("This comparison requires FP32 inputs")
benchmark_report.update({
    **restore_info,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint_source": CHECKPOINT_SOURCE,
    "checkpoint_download": download_info,
    "camera_poses": list(CAMERA_POSES),
    "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
    "preprocessing": {
        "resize": list(MODEL_INPUT_SIZE),
        "resize_shorter_side": None,
        "center_crop": None,
        "pose_crop_enabled": True,
        "normalize_mean": list(IMAGENET_MEAN),
        "normalize_std": list(IMAGENET_STD),
        "augmentations_enabled": False,
    },
    "dataset_root": str(DATASET_ROOT),
    "validation_condition_counts": dict(condition_counts),
    "manifest_sha256": manifest_sha256,
    "dataset_matches_source_manifest": dataset_matches_source_manifest,
    "pin_memory": BENCHMARK_PIN_MEMORY,
    "num_workers": BENCHMARK_NUM_WORKERS,
})
print(json.dumps(benchmark_report, indent=2, allow_nan=False))


In [ ]:
from __future__ import annotations

# Defaults also allow this cell to run in an already-fitted live session,
# without rerunning Configuration and changing the original run/output identity.
ENERGY_ENABLED = globals().get("ENERGY_ENABLED", True)
ENERGY_METHOD = globals().get("ENERGY_METHOD", "auto")
ENERGY_REPEATS = globals().get("ENERGY_REPEATS", 3)
ENERGY_DURATION_SECONDS = globals().get("ENERGY_DURATION_SECONDS", 30.0)
ENERGY_WARMUP_SECONDS = globals().get("ENERGY_WARMUP_SECONDS", 5.0)
ENERGY_POWER_SAMPLE_INTERVAL_SECONDS = globals().get("ENERGY_POWER_SAMPLE_INTERVAL_SECONDS", 0.1)

if ENERGY_ENABLED:
    print(
        f"Measuring GPU energy: {ENERGY_REPEATS} x {ENERGY_DURATION_SECONDS:g} s "
        f"after {ENERGY_WARMUP_SECONDS:g} s warm-up; predict-only, batch 1, FP32"
    )
    energy_report = measure_anomaly_inference_energy(
        detector, benchmark_loader, device=DEVICE,
        repeats=ENERGY_REPEATS,
        duration_seconds=ENERGY_DURATION_SECONDS,
        warmup_seconds=ENERGY_WARMUP_SECONDS,
        sample_interval_seconds=ENERGY_POWER_SAMPLE_INTERVAL_SECONDS,
        method=ENERGY_METHOD,
    )
else:
    energy_report = {
        "schema_version": 1, "status": "disabled",
        "reason": "ENERGY_ENABLED=False", "method": None,
        "energy_joules_per_image": None,
        "average_gpu_power_watts": None,
        "energy_joules_per_image_std": None,
        "runs": [], "warnings": [],
    }
energy_report.update({
    "source_run_id": restore_info["source_run_id"],
    "checkpoint_path": restore_info["checkpoint_path"],
    "manifest_sha256": manifest_sha256,
    "camera_poses": list(CAMERA_POSES),
    "pose_crops": {pose: list(crop) for pose, crop in POSE_CROPS.items()},
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "training_performed": False,
    "calibration_performed": False,
})
benchmark_report["gpu_energy_measurement"] = energy_report
benchmark_report["energy_joules_per_image"] = energy_report["energy_joules_per_image"]
benchmark_report["energy_note"] = (
    "Measured device-wide GPU energy per predict-only inference via "
    + str(energy_report["method"])
    + "; excludes CPU/input transfer/loader; idle is not subtracted"
    if energy_report["status"] == "measured"
    else "GPU energy not measured completely: " + str(energy_report.get("reason"))
)
print({
    "energy_status": energy_report["status"],
    "energy_method": energy_report["method"],
    "gpu_energy_joules_per_image": energy_report["energy_joules_per_image"],
    "average_gpu_power_watts": energy_report["average_gpu_power_watts"],
    "energy_joules_per_image_std": energy_report["energy_joules_per_image_std"],
    "warnings": energy_report["warnings"],
    "reason": energy_report.get("reason"),
})
if energy_report["status"] not in {"measured", "disabled"}:
    print("WARNING: energy could not be measured reliably; inspect inference_energy_batch1.json")

print("Updated benchmark JSON (includes the energy phase):")
print(json.dumps(benchmark_report, indent=2, allow_nan=False))


In [ ]:
from __future__ import annotations

benchmark_run_id = (
    "supersimplenet_256x256_A_crop720_perlin02_unrestricted_top1pct_sigma4_efficiency__"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
OUTPUT_DIR = OUTPUT_ROOT / benchmark_run_id
benchmark_report["benchmark_run_id"] = benchmark_run_id
energy_report["benchmark_run_id"] = benchmark_run_id
report_path = save_json_atomic(benchmark_report, OUTPUT_DIR / "inference_benchmark_batch1.json")
energy_path = save_json_atomic(energy_report, OUTPUT_DIR / "inference_energy_batch1.json")
memory = benchmark_report["cuda_memory"] or {}
predict_memory = benchmark_report["predict_only_cuda_memory"] or {}
summary_row = {
    "benchmark_run_id": benchmark_run_id,
    "source_run_id": benchmark_report["source_run_id"],
    "model": benchmark_report["model"],
    "gpu": benchmark_report["gpu_name"],
    "input_height": MODEL_INPUT_SIZE[0],
    "input_width": MODEL_INPUT_SIZE[1],
    "batch_size": benchmark_report["batch_size"],
    "gaussian_sigma": benchmark_report["gaussian_sigma"],
    "image_score_mode": benchmark_report["image_score_mode"],
    "image_score_fraction": benchmark_report["image_score_fraction"],
    "parameters": benchmark_report["total_parameters"],
    "parameters_mib": benchmark_report["parameters_mib"],
    "buffers_mib": benchmark_report["buffers_mib"],
    "checkpoint_mib": benchmark_report["checkpoint_size_bytes"] / 2**20,
    "predict_only_mean_ms": benchmark_report["latency"]["predict_only"]["mean_ms"],
    "predict_only_p95_ms": benchmark_report["latency"]["predict_only"]["p95_ms"],
    "host_to_device_and_predict_mean_ms": benchmark_report["latency"]["host_to_device_and_predict"]["mean_ms"],
    "end_to_end_mean_ms": benchmark_report["latency"]["end_to_end"]["mean_ms"],
    "predict_only_images_per_second": benchmark_report["predict_only_images_per_second"],
    "host_to_device_and_predict_images_per_second": benchmark_report["throughput_images_per_second"],
    "end_to_end_images_per_second": benchmark_report["end_to_end_images_per_second"],
    "cuda_peak_allocated_mib": memory.get("peak_allocated_mib"),
    "cuda_peak_reserved_mib": memory.get("peak_reserved_mib"),
    "cuda_incremental_peak_allocated_mib": memory.get("incremental_peak_allocated_mib"),
    "predict_only_cuda_peak_allocated_mib": predict_memory.get("peak_allocated_mib"),
    "energy_joules_per_image": benchmark_report["energy_joules_per_image"],
    "energy_status": energy_report["status"],
    "energy_method": energy_report["method"],
    "average_gpu_power_watts": energy_report["average_gpu_power_watts"],
    "energy_joules_per_image_std": energy_report["energy_joules_per_image_std"],
}
summary_path = OUTPUT_DIR / "inference_benchmark_batch1.csv"
with summary_path.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(summary_row))
    writer.writeheader()
    writer.writerow(summary_row)
print(f"JSON report: {report_path}")
print(f"CSV summary: {summary_path}")
print(f"Energy report: {energy_path}")
print("No training, accuracy evaluation, checkpoint overwrite or Drive upload was performed.")


## Saved reports

The output directory contains:

- `inference_benchmark_batch1.json`: latency, throughput, parameters/buffers, checkpoint size, CUDA memory, exact saved/restored configuration, input pipeline, environment, and the complete energy measurement.
- `inference_benchmark_batch1.csv`: one-row summary including GPU J/image, mean GPU power, repeat variability, sensor backend and selected image-scoring mode.
- `inference_energy_batch1.json`: energy protocol, status/reason, sensor details and raw repeated windows.

If NVML cannot measure energy reliably, the energy value remains null and the reason is saved. No TDP-based or latency-based energy estimate is substituted.

Compare predict-only energy and latency on the same GPU with the EfficientAD, TinyGLASS and PatchCore reports. This notebook does not upload outputs, write a new checkpoint, or recompute accuracy.
